# Kaggle training notebook

This notebook trains the `shielded_dueling_double_dqn` agent on Kaggle and saves each run in a dedicated folder.

Output layout:

- `agent/shielded_dueling_double_dqn/runs/<DDMMYYYY-HHMMSS>/`
- `weights/` inside that run folder stores checkpoints
- other artifacts stay in the run folder root
- checkpoints are saved every 50 episodes with the total step count as the filename
- the final checkpoint is saved as `latest.pth`
- TensorBoard logs are written to `tensorboard/`


In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

REPO_CANDIDATES = [
    Path('/kaggle/working/Bomberland-GDGoC-AI-Challenge'),
    Path('/kaggle/input/Bomberland-GDGoC-AI-Challenge'),
    Path.cwd(),
]

for candidate in REPO_CANDIDATES:
    if (candidate / 'agent' / 'shielded_dueling_double_dqn').exists():
        REPO_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError('Could not find the Bomberland repository root.')

os.chdir(REPO_ROOT)
AGENT_DIR = REPO_ROOT / 'agent' / 'shielded_dueling_double_dqn'
if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))

from core import (  # noqa: E402
    ACTION_DIM,
    MAX_BOMB_TIMER,
    SPATIAL_CHANNELS,
    VECTOR_FEATURES,
    NStepAccumulator,
    PrioritizedReplayBuffer,
    RawTransition,
    encode_observation,
)
from opponents import spawn_opponents  # noqa: E402
from pipeline import Learner  # noqa: E402
from reward import compute_reward, final_rank_bonus  # noqa: E402
from utils import seed_everything  # noqa: E402

print(f'Repo root: {REPO_ROOT}')
print(f'Agent dir: {AGENT_DIR}')
print(f'CUDA available: {torch.cuda.is_available()}')


In [ ]:
MAX_PLAYERS = 4

CONFIG = {
    'enemy_type': 'league',
    'num_episodes': 1000,
    'max_steps': 500,
    'seed': 86,
    'save_model': True,
    'pretrained_model': None,
    'opponent_paths': (),
    'batch_size': 128,
    'learning_starts': 10000,
    'train_freq': 4,
    'target_update_interval': 4000,
    'replay_capacity': 200000,
    'n_step': 3,
    'gamma': 0.99,
    'learning_rate': 1e-4,
    'epsilon_start': 1.0,
    'epsilon_final': 0.05,
    'epsilon_decay_steps': 300000,
    'beta_start': 0.4,
    'beta_final': 1.0,
    'beta_growth_steps': 300000,
    'max_grad_norm': 10.0,
    'checkpoint_interval_episodes': 50,
}

def timestamp() -> str:
    return datetime.now().strftime('%d%m%Y-%H%M%S')


def unique_run_dir(root: Path) -> Path:
    root.mkdir(parents=True, exist_ok=True)
    base = root / timestamp()
    if not base.exists():
        return base
    index = 1
    while True:
        candidate = root / f'{base.name}_{index}'
        if not candidate.exists():
            return candidate
        index += 1


def epsilon_by_step(global_step: int) -> float:
    if global_step >= CONFIG['epsilon_decay_steps']:
        return float(CONFIG['epsilon_final'])
    progress = float(global_step) / float(max(1, CONFIG['epsilon_decay_steps']))
    return float(CONFIG['epsilon_start'] - (CONFIG['epsilon_start'] - CONFIG['epsilon_final']) * progress)


def beta_by_step(global_step: int) -> float:
    if global_step >= CONFIG['beta_growth_steps']:
        return float(CONFIG['beta_final'])
    progress = float(global_step) / float(max(1, CONFIG['beta_growth_steps']))
    return float(CONFIG['beta_start'] + (CONFIG['beta_final'] - CONFIG['beta_start']) * progress)


def moving_average(values: list[float], window: int = 10) -> np.ndarray:
    if not values:
        return np.asarray([], dtype=np.float32)
    if len(values) < window:
        return np.asarray(values, dtype=np.float32)
    weights = np.ones(window, dtype=np.float32) / float(window)
    return np.convolve(np.asarray(values, dtype=np.float32), weights, mode='valid')


def safe_save_plot(values: list[float], path: Path, title: str, ylabel: str, window: int | None = None) -> None:
    if not values:
        return
    series = np.asarray(values, dtype=np.float32)
    plt.figure(figsize=(10, 4))
    plt.plot(series, label=title)
    if window is not None and len(values) >= window:
        ma = moving_average(values, window=window)
        plt.plot(np.arange(len(ma)) + window - 1, ma, label=f'moving average ({window})')
    plt.title(title)
    plt.xlabel('Episode')
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path)
    plt.close()


def save_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


In [ ]:
def train() -> dict:
    seed_everything(int(CONFIG['seed']))

    runs_root = AGENT_DIR / 'runs'
    run_dir = unique_run_dir(runs_root)
    weights_dir = run_dir / 'weights'
    tensorboard_dir = run_dir / 'tensorboard'
    run_dir.mkdir(parents=True, exist_ok=False)
    weights_dir.mkdir(parents=True, exist_ok=False)
    tensorboard_dir.mkdir(parents=True, exist_ok=False)

    save_json(run_dir / 'config.json', CONFIG)

    writer = SummaryWriter(log_dir=str(tensorboard_dir))
    metrics_path = run_dir / 'metrics.csv'
    metrics_file = metrics_path.open('w', newline='', encoding='utf-8')
    metrics_writer = csv.DictWriter(
        metrics_file,
        fieldnames=[
            'episode',
            'global_step',
            'episode_reward',
            'episode_loss',
            'win',
            'epsilon',
            'replay_size',
            'episode_steps',
            'seconds',
        ],
    )
    metrics_writer.writeheader()

    learner = Learner(
        spatial_channels=SPATIAL_CHANNELS,
        vector_dim=VECTOR_FEATURES,
        action_dim=ACTION_DIM,
        learning_rate=float(CONFIG['learning_rate']),
        gamma=float(CONFIG['gamma']),
        load_model=CONFIG['pretrained_model'],
    )
    replay = PrioritizedReplayBuffer(
        capacity=int(CONFIG['replay_capacity']),
        spatial_shape=(SPATIAL_CHANNELS, 13, 13),
        vector_dim=VECTOR_FEATURES,
        alpha=0.6,
    )
    nstep = NStepAccumulator(n_step=int(CONFIG['n_step']), gamma=float(CONFIG['gamma']))

    from engine.game import BomberEnv

    env = BomberEnv(max_steps=int(CONFIG['max_steps']), seed=int(CONFIG['seed']))
    history = {'rewards': [], 'losses': [], 'wins': []}
    global_step = 0
    start_time = time.time()

    episode_bar = tqdm(
        total=int(CONFIG['num_episodes']),
        desc='Training',
        unit='episode',
        dynamic_ncols=True,
    )

    try:
        for episode in range(1, int(CONFIG['num_episodes']) + 1):
            episode_start = time.time()
            obs = env.reset(seed=int(CONFIG['seed']) + episode)
            prev_obs = None
            done = False
            episode_reward = 0.0
            episode_steps = 0
            episode_losses: list[float] = []
            epsilon = epsilon_by_step(global_step)
            survival_steps = np.zeros(MAX_PLAYERS, dtype=np.float32)
            opponents = spawn_opponents(str(CONFIG['enemy_type']), tuple(CONFIG['opponent_paths']))

            while not done:
                bundle = encode_observation(
                    obs,
                    agent_id=0,
                    step_index=global_step,
                    max_steps=int(CONFIG['max_steps']),
                )
                epsilon = epsilon_by_step(global_step)
                action = learner.act(bundle, epsilon)

                actions = [action]
                for pid in range(1, MAX_PLAYERS):
                    actions.append(opponents[pid].act(obs))

                next_obs, terminated, truncated = env.step(actions)
                done = bool(terminated or truncated)
                reward = compute_reward(prev_obs, next_obs, agent_id=0)
                survival_steps += np.asarray(next_obs['players'], dtype=np.int32)[:, 2].astype(np.float32)
                if done:
                    reward += final_rank_bonus(next_obs, agent_id=0, survival_steps=survival_steps)

                next_bundle = encode_observation(
                    next_obs,
                    agent_id=0,
                    step_index=global_step + 1,
                    max_steps=int(CONFIG['max_steps']),
                )
                for entry in nstep.push(
                    RawTransition(
                        spatial=bundle.spatial,
                        vector=bundle.vector,
                        action=int(action),
                        reward=float(reward),
                        next_spatial=next_bundle.spatial,
                        next_vector=next_bundle.vector,
                        done=done,
                    )
                ):
                    replay.add(entry)

                episode_reward += float(reward)
                episode_steps += 1
                global_step += 1
                learner.global_step = global_step
                learner.epsilon = epsilon

                if len(replay) >= int(CONFIG['learning_starts']) and global_step % int(CONFIG['train_freq']) == 0:
                    beta = beta_by_step(global_step)
                    batch = replay.sample(batch_size=int(CONFIG['batch_size']), beta=beta)
                    loss, td_errors = learner.optimize(batch, max_grad_norm=float(CONFIG['max_grad_norm']))
                    replay.update_priorities(batch['indices'], td_errors)
                    episode_losses.append(float(loss))

                if global_step % int(CONFIG['target_update_interval']) == 0:
                    learner.sync_target()

                prev_obs = obs
                obs = next_obs

            final_players = np.asarray(obs['players'], dtype=np.int32)
            win = 1.0 if int(final_players[0][2]) == 1 and int(np.sum(final_players[:, 2])) == 1 else 0.0
            episode_loss = float(np.mean(episode_losses)) if episode_losses else None
            elapsed = time.time() - episode_start

            history['rewards'].append(float(episode_reward))
            history['wins'].append(float(win))
            if episode_loss is not None:
                history['losses'].append(float(episode_loss))

            writer.add_scalar('train/reward', float(episode_reward), episode)
            writer.add_scalar('train/win', float(win), episode)
            writer.add_scalar('train/epsilon', float(epsilon), episode)
            writer.add_scalar('train/replay_size', len(replay), episode)
            writer.add_scalar('train/episode_steps', episode_steps, episode)
            writer.add_scalar('train/global_step', global_step, episode)
            if episode_loss is not None:
                writer.add_scalar('train/loss', float(episode_loss), episode)

            metrics_writer.writerow(
                {
                    'episode': episode,
                    'global_step': global_step,
                    'episode_reward': float(episode_reward),
                    'episode_loss': '' if episode_loss is None else float(episode_loss),
                    'win': float(win),
                    'epsilon': float(epsilon),
                    'replay_size': len(replay),
                    'episode_steps': episode_steps,
                    'seconds': round(elapsed, 3),
                }
            )
            metrics_file.flush()

            loss_text = '-' if episode_loss is None else f'{episode_loss:.4f}'
            tqdm.write(
                f"Episode {episode}/{int(CONFIG['num_episodes'])} | reward={episode_reward:.3f} | "
                f"loss={loss_text} | win={int(win)} | eps={epsilon:.3f} | "
                f"replay={len(replay)} | steps={episode_steps} | {elapsed:.1f}s"
            )
            episode_bar.update(1)
            episode_bar.set_postfix(
                reward=f'{episode_reward:.3f}',
                loss=loss_text,
                win=int(win),
                eps=f'{epsilon:.3f}',
                replay=len(replay),
                steps=episode_steps,
            )

            if episode % int(CONFIG['checkpoint_interval_episodes']) == 0:
                checkpoint_path = weights_dir / f'{global_step}.pth'
                learner.save(
                    checkpoint_path,
                    metadata={
                        'run_dir': str(run_dir),
                        'episode': episode,
                        'global_step': global_step,
                        'checkpoint_kind': 'periodic',
                    },
                )

        latest_path = weights_dir / 'latest.pth'
        learner.save(
            latest_path,
            metadata={
                'run_dir': str(run_dir),
                'episode': int(CONFIG['num_episodes']),
                'global_step': global_step,
                'checkpoint_kind': 'latest',
            },
        )

        safe_save_plot(history['losses'], run_dir / 'loss.png', 'Training Loss', 'Loss')
        safe_save_plot(history['rewards'], run_dir / 'rewards.png', 'Episode Reward', 'Reward')
        win_rates = list(np.cumsum(history['wins']) / np.arange(1, len(history['wins']) + 1)) if history['wins'] else []
        safe_save_plot(win_rates, run_dir / 'win_rates.png', 'Win Rate', 'Win Rate')
        safe_save_plot(history['rewards'], run_dir / 'moving_average.png', 'Moving Average Reward', 'Reward', window=10)

        writer.flush()
        writer.close()
        metrics_file.close()

        total_elapsed = time.time() - start_time
        summary = {
            'run_dir': str(run_dir),
            'weights_dir': str(weights_dir),
            'tensorboard_dir': str(tensorboard_dir),
            'latest_checkpoint': str(latest_path),
            'episodes': int(CONFIG['num_episodes']),
            'global_step': global_step,
            'elapsed_seconds': round(total_elapsed, 2),
        }
        save_json(run_dir / 'summary.json', summary)
        return summary
    finally:
        try:
            writer.close()
        except Exception:
            pass
        try:
            metrics_file.close()
        except Exception:
            pass


summary = train()
summary


In [ ]:
TENSORBOARD_LOGDIR = str(AGENT_DIR / 'runs')
print(f'TensorBoard logdir: {TENSORBOARD_LOGDIR}')

try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'tensorboard')
        ip.run_line_magic('tensorboard', f'--logdir {TENSORBOARD_LOGDIR}')
except Exception as exc:
    print(f'Could not launch TensorBoard automatically: {exc}')
